# schedule_manifest_source_v1

目标：验证 `src/schedule` 的 dashboard refresh payload 会记录 dashboard export 使用的 manifest 来源，便于通知、审计和页面追踪。

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO = Path('/Users/rziiiii/dev/workspace')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.dashboard import write_model_matrix_dashboard_export
from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = Path('research/schedule_manifest_source_v1')
MATRIX_CARD = CARD / 'matrix_card'
DASHBOARD_EXPORT = CARD / 'dashboard_export.json'
LOCK_DIR = CARD / 'locks'
CARD.mkdir(parents=True, exist_ok=True)
MATRIX_CARD.mkdir(parents=True, exist_ok=True)
{
    'card': str(CARD),
    'matrix_card': str(MATRIX_CARD),
    'dashboard_export': str(DASHBOARD_EXPORT),
}


{'card': 'research/schedule_manifest_source_v1',
 'matrix_card': 'research/schedule_manifest_source_v1/matrix_card',
 'dashboard_export': 'research/schedule_manifest_source_v1/dashboard_export.json'}

## 构造 manifest-first card

自定义 sidecar 文件名由 `analysis_manifest.json` 指定，schedule 的 dashboard export 默认函数会通过 `src/dashboard` 读取。

In [2]:
def write_manifest_first_card(card_dir: Path) -> None:
    card_dir.mkdir(parents=True, exist_ok=True)
    (card_dir / 'summary_manifest.json').write_text(
        json.dumps(
            {'schema_version': 'analysis.report.v1', 'status': 'schedule_manifest_source_completed', 'model_count': 1},
            ensure_ascii=False,
            indent=2,
        ),
        encoding='utf-8',
    )
    pd.DataFrame([{'rank': 1, 'model': 'ridge_mse', 'return_pct': 2.0, 'relative_return_pct': 1.0, 'stability_score': 3.0}]).to_csv(card_dir / 'comparison_manifest.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'regime': 'bull', 'return_pct': 2.0}]).to_csv(card_dir / 'regime_manifest.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'window': 'test', 'return_pct': 2.0}]).to_csv(card_dir / 'window_manifest.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'return_kind': 'strategy_return', 'annualized_return': 0.22}]).to_csv(card_dir / 'risk_manifest.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'segment_type': 'all', 'segment': 'all', 'diagnosis': '基准模型。'}]).to_csv(card_dir / 'deltas_manifest.csv', index=False)
    manifest = {
        'artifact_id': 'schedule_manifest_source_v1',
        'artifact_type': 'analysis_report',
        'producer': 'src.analysis.exports',
        'schema_version': 'artifact.v1',
        'paths': {
            'summary': 'summary_manifest.json',
            'model_comparison': 'comparison_manifest.csv',
            'regime_metrics': 'regime_manifest.csv',
            'window_metrics': 'window_manifest.csv',
            'qlib_risk_metrics': 'risk_manifest.csv',
            'model_effect_deltas': 'deltas_manifest.csv',
        },
        'metadata': {'analysis_report_schema_version': 'analysis.report.v1'},
        'manifest_hash': 's' * 64,
    }
    (card_dir / 'analysis_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

write_manifest_first_card(MATRIX_CARD)
sorted(path.name for path in MATRIX_CARD.iterdir())


['analysis_manifest.json',
 'comparison_manifest.csv',
 'deltas_manifest.csv',
 'regime_manifest.csv',
 'risk_manifest.csv',
 'summary_manifest.json',
 'window_manifest.csv']

## 运行 schedule refresh

为了聚焦 schedule 与 dashboard 的衔接，matrix_runner 只确认 card 已存在；dashboard_exporter 使用默认 `write_model_matrix_dashboard_export`。

In [3]:
def matrix_runner(**kwargs):
    card_dir = Path(kwargs['card_dir'])
    assert (card_dir / 'analysis_manifest.json').exists()
    return {'status': 'schedule_manifest_source_completed', 'variant_count': 1}

config = ModelDashboardPipelineConfig(
    name='schedule_manifest_source_v1',
    card_dir=MATRIX_CARD,
    dashboard_output_path=DASHBOARD_EXPORT,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse',),
)
result = run_model_dashboard_refresh_once(
    config,
    lock_dir=LOCK_DIR,
    matrix_runner=matrix_runner,
    dashboard_exporter=write_model_matrix_dashboard_export,
)
assert result.status == 'success'
assert result.payload['dashboard_manifest_kind'] == 'analysis_manifest'
assert result.payload['dashboard_manifest_hash'] == 's' * 64
assert result.payload['dashboard_manifest_producer'] == 'src.analysis.exports'
result.payload


{'matrix_card': 'research/schedule_manifest_source_v1/matrix_card',
 'dashboard_export': 'research/schedule_manifest_source_v1/dashboard_export.json',
 'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'dashboard_artifact_id': 'schedule_manifest_source_v1',
 'dashboard_manifest_kind': 'analysis_manifest',
 'dashboard_manifest_path': 'research/schedule_manifest_source_v1/matrix_card/analysis_manifest.json',
 'dashboard_manifest_hash': 'ssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssss',
 'dashboard_manifest_producer': 'src.analysis.exports',
 'dashboard_page': '',
 'dashboard_page_schema_version': '',
 'dashboard_view_model_schema_version': '',
 'dashboard_page_model_card_count': 0,
 'top_model': 'ridge_mse',
 'attribution_card': '',
 'attribution_status': 'not_provided',
 'matrix_status': 'schedule_manifest_source_completed',
 'variant_count': 1,
 'attempts': 1}

## 检查 dashboard export

dashboard export 自身也保留 `manifest_source`，schedule payload 与 export 可互相校验。

In [4]:
dashboard_payload = json.loads(DASHBOARD_EXPORT.read_text(encoding='utf-8'))
assert dashboard_payload['manifest_source']['kind'] == result.payload['dashboard_manifest_kind']
assert dashboard_payload['manifest_source']['manifest_hash'] == result.payload['dashboard_manifest_hash']
{
    'dashboard_artifact_id': dashboard_payload['artifact_id'],
    'manifest_source': dashboard_payload['manifest_source'],
    'top_model': dashboard_payload['top_models'][0]['model'],
}


{'dashboard_artifact_id': 'schedule_manifest_source_v1',
 'manifest_source': {'kind': 'analysis_manifest',
  'path': 'research/schedule_manifest_source_v1/matrix_card/analysis_manifest.json',
  'schema_version': 'artifact.v1',
  'manifest_hash': 'ssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssss',
  'producer': 'src.analysis.exports'},
 'top_model': 'ridge_mse'}

## 写出本实验摘要

根目录摘要记录 schedule payload 中的 manifest 审计字段。

In [5]:
run_summary = {
    'schema_version': 'research.schedule_manifest_source_v1',
    'status': result.status,
    'matrix_card': str(MATRIX_CARD),
    'dashboard_export': str(DASHBOARD_EXPORT),
    'dashboard_artifact_id': result.payload['dashboard_artifact_id'],
    'dashboard_manifest_kind': result.payload['dashboard_manifest_kind'],
    'dashboard_manifest_path': result.payload['dashboard_manifest_path'],
    'dashboard_manifest_hash': result.payload['dashboard_manifest_hash'],
    'dashboard_manifest_producer': result.payload['dashboard_manifest_producer'],
    'conclusion': 'src/schedule 已将 dashboard 使用的 manifest 来源写入 refresh payload。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8')
run_summary


{'schema_version': 'research.schedule_manifest_source_v1',
 'status': 'success',
 'matrix_card': 'research/schedule_manifest_source_v1/matrix_card',
 'dashboard_export': 'research/schedule_manifest_source_v1/dashboard_export.json',
 'dashboard_artifact_id': 'schedule_manifest_source_v1',
 'dashboard_manifest_kind': 'analysis_manifest',
 'dashboard_manifest_path': 'research/schedule_manifest_source_v1/matrix_card/analysis_manifest.json',
 'dashboard_manifest_hash': 'ssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssss',
 'dashboard_manifest_producer': 'src.analysis.exports',
 'conclusion': 'src/schedule 已将 dashboard 使用的 manifest 来源写入 refresh payload。'}

## 中文结论

- 做了什么：验证 `src/schedule` 的模型 dashboard refresh 会透传 dashboard 使用的 manifest 来源。
- 核心指标：本轮输出 `dashboard_artifact_id`、`dashboard_manifest_kind`、`dashboard_manifest_path`、`dashboard_manifest_hash`、`dashboard_manifest_producer`，并与 `dashboard_export.json` 中的 `manifest_source` 一致。
- regime 差异：本轮不重新评估模型 regime 表现，重点是调度层审计字段；dashboard export 仍保留 regime/window/risk/delta sidecar。
- 是否晋升：这是框架能力，不是交易策略晋升；它让定时刷新、通知和页面追踪可以定位统一 manifest。
- 风险：旧 fixed-sidecar card 会显示 `fixed_sidecars` 来源且无 manifest hash；后续需要补一个旧 card manifest 迁移工具。
- 下一步：继续补 `src/core` 或 `scripts` 的 manifest backfill 工具，给旧 research card 生成 `analysis_manifest.json`。